# GuppyLM カタカナ版

カタカナで会話する ~9M パラメータの LLM をゼロから学習します。

**このノートブックでできること:**
1. タグ付き単語辞書 + 文法パターンでカタカナ会話データを 60,000 サンプル生成
2. BPE トークナイザーを学習
3. 6層トランスフォーマー (8.7M params) を学習
4. テストケースで推論を体験

**アーキテクチャ:** 6層, 384 dim, 6 heads, ReLU FFN, LayerNorm, 4096 vocab

**所要時間:** T4 GPU で約 5 分

**結果:** 水、エサ、ヒカリのことしか話さないカタカナのサカナ

## 1. セットアップ

リポジトリをクローンし、依存ライブラリをインストールします。

In [ ]:
!pip install -q torch tokenizers tqdm numpy
!git clone https://github.com/high-u/guppylm.git

import sys, os
sys.path.insert(0, '/content/guppylm')
os.chdir('/content/guppylm')

import torch
print(f'PyTorch {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'作業ディレクトリ: {os.getcwd()}')

## 2. データ生成

カタカナの会話データを生成します。

`generate_data.py` はタグ付き単語辞書 (約 450 語) と文法パターン (P1〜P8) を使って、
60 トピック × 1,000 サンプル = 60,000 の会話データを組み立てます。

各サンプルは ChatML 形式:
```
<|im_start|>user
コンニチハ グッピー<|im_end|>
<|im_start|>assistant
ヤッホー。エサ ハ オイシイ。アサ ガ キタ。<|im_end|>
```

In [ ]:
from guppylm.generate_data import generate_dataset
generate_dataset(60000)

In [ ]:
import json
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors

# 生成されたテキストを読み込み
texts = []
for path in ['data/train.jsonl', 'data/eval.jsonl']:
    with open(path) as f:
        for line in f:
            texts.append(json.loads(line)['text'])

print(f'テキスト数: {len(texts):,}')

# BPE トークナイザーを学習
tokenizer = Tokenizer(models.BPE())
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
tokenizer.decoder = decoders.ByteLevel()

trainer = trainers.BpeTrainer(
    vocab_size=4096,
    special_tokens=['<pad>', '<|im_start|>', '<|im_end|>'],
    min_frequency=2,
    show_progress=True,
)
tokenizer.train_from_iterator(texts, trainer)
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.save('data/tokenizer.json')
print(f'トークナイザー: {tokenizer.get_vocab_size()} トークン')

# プレビュー
with open('data/train.jsonl') as f:
    sample = json.loads(f.readline())
print(f'\nサンプル ({sample["category"]}):\n{sample["text"]}')

## 3. モデル確認

モデルをビルドし、パラメータ数とフォワードパスを確認します。

In [ ]:
from guppylm.config import GuppyConfig
from guppylm.model import GuppyLM
import torch

config = GuppyConfig()
model = GuppyLM(config)
print(model.param_summary())
print(f'  層数: {config.n_layers}, ヘッド数: {config.n_heads}, FFN: {config.ffn_hidden}')
print(f'  語彙数: {config.vocab_size}, 最大系列長: {config.max_seq_len}')

# ダミーフォワードパス
x = torch.randint(0, config.vocab_size, (2, 32))
logits, _ = model(x)
print(f'  フォワードパス: {x.shape} -> {logits.shape} OK')
del model

## 4. 学習

10,000 ステップのコサイン LR スケジュールで学習します。T4 GPU で約 2〜3 分。

モデルは以下を学習します:
- カタカナで短い文を返す
- サカナのキャラクターを保つ
- 60 の異なるトピックに対応する
- 適切なタイミングで生成を停止する (`<|im_end|>` トークン)

In [ ]:
from guppylm.train import train
train()

## 5. 推論テスト

学習したモデルで推論を実行します。
`eval_cases.py` に定義された 16 のテストケースで、各トピックの応答を確認します。

In [ ]:
from guppylm.inference import GuppyInference
from guppylm.eval_cases import get_eval_cases
import torch

device = 'cuda' if torch.cuda.is_available() else 'cpu'
engine = GuppyInference('checkpoints/best_model.safetensors', 'data/tokenizer.json', device=device)

def chat(prompt):
    r = engine.chat_completion([{'role': 'user', 'content': prompt}], max_tokens=64)
    return r['choices'][0]['message'].get('content', '').strip()

cases = get_eval_cases()

print(f'{"カテゴリ":<14s}  {"ユーザー":<36s}  グッピー')
print('=' * 110)
for case in cases:
    reply = chat(case['prompt'])
    print(f'{case["category"]:<14s}  {case["prompt"]:<36s}  {reply[:128]}')